# Northstar Retail Databricks practical

Run cells in order. Complete the environment configuration before using compute.

First complete Tasks A and B in TP_Day_03: create your own workspace, create the managed source volume, upload the five files and pass the source-file checkpoint. Then copy your values from Environment_Info.


## Configuration


In [0]:
from pyspark.sql import functions as F
from pathlib import Path
import re

CATALOG = "YOUR_CATALOG"
SCHEMA = "northstar_YOUR_NAME"
VOLUME = "source_files"
SOURCE_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/day03"
BRONZE = f"`{CATALOG}`.`{SCHEMA}`.`sales_bronze`"
SILVER = f"`{CATALOG}`.`{SCHEMA}`.`sales_silver`"
GOLD = f"`{CATALOG}`.`{SCHEMA}`.`sales_by_store`"
print(SOURCE_ROOT)

/Volumes/YOUR_CATALOG/northstar_YOUR_NAME/source_files/day03


## Load sales


In [0]:
sales_schema = (
    "SaleID STRING, SaleDate STRING, StoreID STRING, "
    "ProductID STRING, Quantity STRING, UnitPrice STRING"
)
sales_df = (spark.read.option("header", True)
            .option("mode", "FAILFAST")
            .schema(sales_schema)
            .csv(f"{SOURCE_ROOT}/sales.csv"))
sales_df.printSchema()
sales_df.show(8, truncate=False)
print(sales_df.count())

root
 |-- SaleID: string (nullable = true)
 |-- SaleDate: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)

+------+----------+-------+---------+--------+---------+
|SaleID|SaleDate  |StoreID|ProductID|Quantity|UnitPrice|
+------+----------+-------+---------+--------+---------+
|T0001 |2026-09-21|S01    |P01      |1       |4.50     |
|T0002 |2026-09-21|S01    |P02      |2       |12.00    |
|T0003 |2026-09-21|S01    |P03      |3       |7.25     |
|T0004 |2026-09-21|S01    |P04      |4       |19.90    |
|T0005 |2026-09-21|S01    |P05      |5       |3.75     |
|T0006 |2026-09-21|S01    |P06      |1       |8.40     |
|T0007 |2026-09-22|S01    |P01      |2       |4.50     |
|T0008 |2026-09-22|S01    |P02      |3       |12.00    |
+------+----------+-------+---------+--------+---------+
only showing top 8 rows
112


## Bronze


In [0]:
(sales_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(BRONZE))
bronze_df = spark.table(BRONZE)
bronze_df.printSchema()
print(bronze_df.count())

root
 |-- SaleID: string (nullable = true)
 |-- SaleDate: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)

112


## Silver cleaning


In [0]:
dedup_df = bronze_df.dropDuplicates()
print("after dedup:", dedup_df.count())

typed_df = (dedup_df
    .withColumn("Quantity",  F.expr("try_cast(Quantity as int)"))
    .withColumn("UnitPrice", F.expr("try_cast(UnitPrice as decimal(12,2))"))
    .withColumn("SaleDate",  F.expr("try_cast(SaleDate as date)")))

silver_df = typed_df.filter(
    F.col("Quantity").isNotNull() & (F.col("Quantity") > 0) &
    F.col("UnitPrice").isNotNull() & (F.col("UnitPrice") > 0) &
    F.col("SaleDate").isNotNull()
)
print("silver rows:", silver_df.count())

(silver_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(SILVER))
spark.table(SILVER).printSchema()

after dedup: 106
silver rows: 96
root
 |-- SaleID: string (nullable = true)
 |-- SaleDate: date (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(12,2) (nullable = true)



## Load products


In [0]:
products_df = (spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema("ProductID STRING, ProductName STRING, Category STRING")
    .csv(f"{SOURCE_ROOT}/products.csv"))
products_df.show(truncate=False)
products_df.printSchema()
print(products_df.count(), products_df.select("ProductID").distinct().count())

+---------+------------+-----------+
|ProductID|ProductName |Category   |
+---------+------------+-----------+
|P01      |Notebook A5 |Stationery |
|P02      |Desk lamp   |Home       |
|P03      |Water bottle|Accessories|
|P04      |Backpack    |Accessories|
|P05      |Pen set     |Stationery |
|P06      |Coffee mug  |Home       |
+---------+------------+-----------+

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)

6 6


## Load stores


In [0]:
stores_df = (spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema("StoreID STRING, StoreName STRING, Region STRING")
    .csv(f"{SOURCE_ROOT}/stores.csv"))
stores_df.show(truncate=False)
stores_df.printSchema()
print(stores_df.count(), stores_df.select("StoreID").distinct().count())

+-------+------------+--------------------+
|StoreID|StoreName   |Region              |
+-------+------------+--------------------+
|S01    |Paris Centre|Ile-de-France       |
|S02    |Versailles  |Ile-de-France       |
|S03    |Lyon Centre |Auvergne-Rhone-Alpes|
|S04    |Grenoble    |Auvergne-Rhone-Alpes|
+-------+------------+--------------------+

root
 |-- StoreID: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- Region: string (nullable = true)

4 4


## Join data


In [0]:
print("unmatched products:", silver_df.join(products_df, "ProductID", "left_anti").count())
print("unmatched stores:", silver_df.join(stores_df, "StoreID", "left_anti").count())

enriched_df = (silver_df
    .join(products_df, "ProductID", "inner")
    .join(stores_df, "StoreID", "inner"))
print("enriched rows:", enriched_df.count())
enriched_df.show(5, truncate=False)

unmatched products: 0
unmatched stores: 0
enriched rows: 96
+-------+---------+------+----------+--------+---------+------------+-----------+------------+-------------+
|StoreID|ProductID|SaleID|SaleDate  |Quantity|UnitPrice|ProductName |Category   |StoreName   |Region       |
+-------+---------+------+----------+--------+---------+------------+-----------+------------+-------------+
|S01    |P01      |T0001 |2026-09-21|1       |4.50     |Notebook A5 |Stationery |Paris Centre|Ile-de-France|
|S01    |P01      |T0007 |2026-09-22|2       |4.50     |Notebook A5 |Stationery |Paris Centre|Ile-de-France|
|S01    |P03      |T0009 |2026-09-22|4       |7.25     |Water bottle|Accessories|Paris Centre|Ile-de-France|
|S01    |P05      |T0011 |2026-09-22|1       |3.75     |Pen set     |Stationery |Paris Centre|Ile-de-France|
|S02    |P03      |T0039 |2026-09-23|1       |7.25     |Water bottle|Accessories|Versailles  |Ile-de-France|
+-------+---------+------+----------+--------+---------+------------

## Revenue


In [0]:
revenue_df = enriched_df.withColumn(
    "Revenue", (F.col("Quantity") * F.col("UnitPrice")).cast("decimal(18,2)")
)
revenue_df.select("SaleID", "Quantity", "UnitPrice", "Revenue").orderBy("SaleID").show(6)
print(revenue_df.count())

+------+--------+---------+-------+
|SaleID|Quantity|UnitPrice|Revenue|
+------+--------+---------+-------+
| T0001|       1|     4.50|   4.50|
| T0002|       2|    12.00|  24.00|
| T0003|       3|     7.25|  21.75|
| T0004|       4|    19.90|  79.60|
| T0005|       5|     3.75|  18.75|
| T0006|       1|     8.40|   8.40|
+------+--------+---------+-------+
only showing top 6 rows
96


## Gold aggregation


In [0]:
gold_df = (revenue_df.groupBy("StoreID", "StoreName", "Region").agg(
        F.count("*").alias("TransactionCount"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("Revenue").cast("decimal(20,2)").alias("TotalRevenue"))
    .orderBy("StoreID"))

(gold_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD))

gold_saved = spark.table(GOLD)
gold_saved.show(truncate=False)
gold_saved.printSchema()

+-------+------------+--------------------+----------------+-------------+------------+
|StoreID|StoreName   |Region              |TransactionCount|TotalQuantity|TotalRevenue|
+-------+------------+--------------------+----------------+-------------+------------+
|S01    |Paris Centre|Ile-de-France       |24              |70           |671.30      |
|S02    |Versailles  |Ile-de-France       |24              |74           |680.00      |
|S03    |Lyon Centre |Auvergne-Rhone-Alpes|24              |73           |642.95      |
|S04    |Grenoble    |Auvergne-Rhone-Alpes|24              |72           |686.65      |
+-------+------------+--------------------+----------------+-------------+------------+

root
 |-- StoreID: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- TransactionCount: long (nullable = true)
 |-- TotalQuantity: long (nullable = true)
 |-- TotalRevenue: decimal(20,2) (nullable = true)



## SQL analysis


In [0]:
spark.sql(f"""
SELECT StoreID, StoreName, TotalRevenue
FROM {GOLD}
ORDER BY TotalRevenue DESC, StoreID
""").show(truncate=False)

spark.sql(f"""
SELECT Region,
       SUM(TransactionCount) AS TransactionCount,
       SUM(TotalQuantity) AS TotalQuantity,
       SUM(TotalRevenue) AS TotalRevenue
FROM {GOLD}
GROUP BY Region
ORDER BY Region
""").show(truncate=False)

+-------+------------+------------+
|StoreID|StoreName   |TotalRevenue|
+-------+------------+------------+
|S04    |Grenoble    |686.65      |
|S02    |Versailles  |680.00      |
|S01    |Paris Centre|671.30      |
|S03    |Lyon Centre |642.95      |
+-------+------------+------------+

+--------------------+----------------+-------------+------------+
|Region              |TransactionCount|TotalQuantity|TotalRevenue|
+--------------------+----------------+-------------+------------+
|Auvergne-Rhone-Alpes|48              |145          |1329.60     |
|Ile-de-France       |48              |144          |1351.30     |
+--------------------+----------------+-------------+------------+



## Final validation


In [0]:
validator_path = Path(SOURCE_ROOT) / "Validate_Day_03.py"
exec(compile(validator_path.read_text(encoding="utf-8"),
             str(validator_path), "exec"))
validate_day03(spark, CATALOG, SCHEMA, SOURCE_ROOT)


PASS sales_bronze: schema, rows and values match
PASS sales_silver: schema, rows and values match
PASS sales_by_store: schema, rows and values match
PASS Day 3: Bronze 112; Silver 96; Gold 4; revenue 2680.90
